<div dir="rtl">

# الدرس 53: الضبط الدقيق، وطريقة لورا

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

## بالكود

</div>

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from torch import nn

@dataclass
class GPTConfig:
    vocab_size: int = 50257
    block_size: int = 1024          # context window
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    dropout: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = cfg.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, 4 * cfg.d_model)
        self.proj = nn.Linear(4 * cfg.d_model, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))      # tokens communicate
        x = x + self.mlp(self.ln2(x))       # each token computes
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):                            # GPT-2 style initialization
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than the context window"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                   # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
import os
import time
import urllib.request

if not os.path.exists("tiny_gpt.pt"):
    raise FileNotFoundError("Run the notebook of lesson 49 first: it trains and saves tiny_gpt.pt")
ckpt = torch.load("tiny_gpt.pt")
cfg = GPTConfig(**ckpt["config"])
model = GPT(cfg)
model.load_state_dict(ckpt["model"])
chars = ckpt["chars"]
stoi = {c: i for i, c in enumerate(chars)}

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(URL, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", encoding="utf-8").read()
encode = lambda s: torch.tensor([stoi[c] for c in s])
split = int(0.9 * len(text))
low_train, low_val = encode(text.lower()[:split]), encode(text.lower()[split:])
orig_val = encode(text[split:])

T, B = cfg.block_size, 32
def get_batch(d, g):
    starts = torch.randint(len(d) - T - 1, (B,), generator=g)
    return torch.stack([d[i:i + T] for i in starts]), torch.stack([d[i + 1:i + T + 1] for i in starts])

@torch.no_grad()
def val_loss(d, n=20):
    model.eval()
    g = torch.Generator().manual_seed(1)
    return sum(model(*get_batch(d, g))[1].item() for _ in range(n)) / n

@torch.no_grad()
def generate(prompt, n, seed=0, temperature=0.8):
    model.eval()
    g = torch.Generator().manual_seed(seed)
    idx = [stoi[c] for c in prompt]
    for _ in range(n):
        logits, _ = model(torch.tensor([idx[-T:]]))
        idx.append(torch.multinomial(torch.softmax(logits[0, -1] / temperature, -1), 1, generator=g).item())
    return "".join(chars[i] for i in idx)

def upper_ratio(s):
    letters = [c for c in s if c.isalpha()]
    return sum(c.isupper() for c in letters) / len(letters)

before = generate("\n", 400)
print(f"before: loss on lowercase {val_loss(low_val):.3f} | on original {val_loss(orig_val):.3f} | "
      f"uppercase letters in generated text {upper_ratio(before):.1%}")

<div dir="rtl">

### لورا من الصفر

</div>

In [ ]:
class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update B @ A."""
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                       # freeze the original weights
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))    # zero: start = original model
        self.scale = alpha / r
        self.enabled = True

    def forward(self, x):
        y = self.base(x)
        if self.enabled:
            y = y + (x @ self.A.T) @ self.B.T * self.scale  # x -> r numbers -> d_out
        return y

for p in model.parameters():
    p.requires_grad = False
for blk in model.blocks:
    blk.attn.qkv, blk.attn.proj = LoRALinear(blk.attn.qkv), LoRALinear(blk.attn.proj)
    blk.mlp.fc, blk.mlp.proj = LoRALinear(blk.mlp.fc), LoRALinear(blk.mlp.proj)

trainable = [p for p in model.parameters() if p.requires_grad]
n_train = sum(p.numel() for p in trainable)
n_all = sum(p.numel() for p in model.parameters())
print(f"trainable {n_train:,} of {n_all:,} ({n_train / n_all:.1%})")
print(f"loss right after adding LoRA: {val_loss(orig_val):.3f}  (unchanged: B starts at zero)")

<div dir="rtl">

### التدريب: المحوّل فقط

</div>

In [ ]:
opt = torch.optim.AdamW(trainable, lr=2e-3)            # only A and B are optimized
g = torch.Generator().manual_seed(0)
t0 = time.time()
for step in range(1, 401):
    model.train()
    _, loss = model(*get_batch(low_train, g))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        print(f"step {step}: loss on lowercase {val_loss(low_val):.3f} | "
              f"on original {val_loss(orig_val):.3f} | {time.time() - t0:.0f}s")

after = generate("\n", 400)
print(f"uppercase letters in generated text: {upper_ratio(after):.1%}\n")
print(after[:300])

<div dir="rtl">

### إطفاء المحوّل، ثم الدمج

</div>

In [ ]:
lora_layers = [m for m in model.modules() if isinstance(m, LoRALinear)]
for m in lora_layers:
    m.enabled = False
print(f"adapter OFF: lowercase {val_loss(low_val):.3f} | original {val_loss(orig_val):.3f}  (the original model is back)")

for m in lora_layers:
    m.enabled = True
x = encode("ROMEO:\nWhat light")[None]
with torch.no_grad():
    with_adapter = model(x)[0]
    for m in lora_layers:                                # merge: W <- W + scale * B @ A
        m.base.weight += m.scale * (m.B @ m.A)
        m.enabled = False
    merged = model(x)[0]
print("merged model == model with adapter:", torch.allclose(with_adapter, merged, atol=1e-5))

<div dir="rtl">

### مكتبة الضبط الفعّال

</div>

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import GPT2Config, GPT2LMHeadModel

small = GPT2LMHeadModel(GPT2Config(vocab_size=1000, n_positions=128, n_embd=128, n_layer=4, n_head=4))
lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn", "c_proj"],
                      lora_dropout=0.05, fan_in_fan_out=True)   # GPT-2 stores weights transposed
peft_model = get_peft_model(small, lora_cfg)
peft_model.print_trainable_parameters()

<div dir="rtl">

### نموذج حقيقي مع التكميم

</div>

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(name)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16    # older cards: float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype)
model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                                         task_type="CAUSAL_LM"))
model.print_trainable_parameters()

def encode_example(messages):
    """Tokenize one chat, with the loss only on the assistant's final answer."""
    prompt = tokenizer.apply_chat_template(messages[:-1], add_generation_prompt=True, tokenize=False)
    full = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(prompt_ids) + ids[len(prompt_ids):]      # lesson 51: mask the prompt
    return torch.tensor(ids), torch.tensor(labels)

# then a normal training loop over your examples: forward with labels=..., backward, step,
# and at the end: model.save_pretrained("my-adapter")   # a few megabytes only

<div dir="rtl">

## تمرين: جرّب رتباً مختلفة

أعد التجربة، لكن مع لورا على طبقة الاستعلام والمفتاح والقيمة فقط، وبرتبتين: 2، ثم 16.

1. احسب بيدك عدد المعاملات القابلة للتدريب لكل رتبة.
2. درّب كل واحدة 400 خطوة من نموذج الدرس 49 الأصلي، وقارن الخسارة على النص الصغير، ونسبة الحروف الكبيرة.

</div>

In [ ]:
# اكتب حلّك هنا
